# Feasibility Check
Xác nhận dataset, model và môi trường chạy được trước khi bắt đầu fine-tuning.
Không phải notebook nghiên cứu — chỉ trả lời: có thể triển khai project không?

In [1]:
# Cell 1: Environment
import sys
import torch
import transformers

print(f'Python     : {sys.version}')
print(f'PyTorch    : {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: CUDA không available — training sẽ rất chậm trên CPU')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device     : {DEVICE}')

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python     : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch    : 2.6.0+cu124
Transformers: 5.18.0
CUDA available: True
GPU        : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM total : 8.0 GB
Device     : cuda


In [2]:
# Cell 2: Load data + verify schema
import pandas as pd
import ast
from pathlib import Path

DATA_DIR = Path('../data/raw')

train = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
val   = pd.read_csv(DATA_DIR / 'val.csv',   encoding='utf-8-sig')
test  = pd.read_csv(DATA_DIR / 'test.csv',  encoding='utf-8-sig')

print('=== Split sizes ===')
print(f'train : {train.shape}')
print(f'val   : {val.shape}')
print(f'test  : {test.shape}')

print(f'\n=== Columns ===')
print(train.columns.tolist())

print(f'\n=== Sample (train) ===')
train.head(3)

=== Split sizes ===
train : (16531, 3)
val   : (2066, 3)
test  : (2067, 3)

=== Columns ===
['id', 'text', 'labels']

=== Sample (train) ===


,id,text,labels
0,tik000008,Xem mà ngẫm lại cuộc đời bản thân ta đã trải q...,[12]
1,5743,bức ảnh xuất sắc ❤️,"[2, 8, 3]"
2,32895,"Vừa đẹp trai, vừa tài giỏi. Nhà mặt phố, bố là...","[8, 7]"


In [3]:
# Cell 3: Verify labels
for df, name in [(train, 'train'), (val, 'val'), (test, 'test')]:
    df['labels'] = df['labels'].apply(ast.literal_eval)

# Tất cả label IDs có trong train
all_ids = sorted(set(lbl for labels in train['labels'] for lbl in labels))
print(f'Unique label IDs in train: {len(all_ids)}')
print(f'IDs: {all_ids}')
print(f'Min ID: {min(all_ids)}, Max ID: {max(all_ids)}')

# Số nhãn mỗi câu
for df, name in [(train, 'train'), (val, 'val'), (test, 'test')]:
    n = df['labels'].apply(len)
    print(f'\n{name} — labels/sample: mean={n.mean():.3f}, median={n.median():.1f}, max={n.max()}')

# Kiểm tra ID 27 (neutral) có hay không
print(f'\nNeutral (ID=27) present: {27 in all_ids}')

Unique label IDs in train: 28
IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Min ID: 0, Max ID: 27

train — labels/sample: mean=1.908, median=2.0, max=5

val — labels/sample: mean=1.916, median=2.0, max=5

test — labels/sample: mean=1.908, median=2.0, max=5

Neutral (ID=27) present: True


In [4]:
# Cell 4: Tokenizer smoke test
from transformers import AutoTokenizer

MODEL_NAME = 'uitnlp/visobert'
print(f'Loading tokenizer from {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

test_sentences = [
    'Xem mà ngẫm lại cuộc đời bản thân ta đã trải qua ❤️😂',   # có emoji
    'mn ơi cái này hay ghê vsss, tl cho mik với',               # có teencode
    'Vừa đẹp trai vừa tài giỏi, thật sự không có gì để chê',   # bình thường
]

for sent in test_sentences:
    enc = tokenizer(sent, return_tensors='pt', truncation=True, max_length=128, padding='max_length')
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    non_pad = [t for t in tokens if t != tokenizer.pad_token]
    print(f'\nInput : {sent[:60]}...' if len(sent) > 60 else f'\nInput : {sent}')
    print(f'Tokens: {non_pad}')
    print(f'Length: {len(non_pad)} (incl. [CLS], [SEP])')

Loading tokenizer from uitnlp/visobert...


Tokenizer loaded.

Input : Xem mà ngẫm lại cuộc đời bản thân ta đã trải qua ❤️😂
Tokens: ['<s>', '▁X', 'em', '▁mà', '▁ng', 'ẫm', '▁l', 'ại', '▁cu', 'ộc', '▁đ', 'ời', '▁b', 'ản', '▁th', 'ân', '▁ta', '▁đã', '▁tr', 'ải', '▁qua', '▁❤️', '😂', '</s>']
Length: 24 (incl. [CLS], [SEP])

Input : mn ơi cái này hay ghê vsss, tl cho mik với
Tokens: ['<s>', '▁mn', '▁ơi', '▁cái', '▁này', '▁h', 'ay', '▁ghê', '▁vs', 'ss', ',', '▁tl', '▁cho', '▁m', 'ik', '▁v', 'ới', '</s>']
Length: 18 (incl. [CLS], [SEP])

Input : Vừa đẹp trai vừa tài giỏi, thật sự không có gì để chê
Tokens: ['<s>', '▁V', 'ừa', '▁đ', 'ẹp', '▁tr', 'ai', '▁v', 'ừa', '▁t', 'ài', '▁gi', 'ỏi', ',', '▁th', 'ật', '▁sự', '▁kh', 'ông', '▁có', '▁gì', '▁để', '▁chê', '</s>']
Length: 24 (incl. [CLS], [SEP])


In [5]:
# Cell 5: Model forward pass
import torch.nn as nn
from transformers import AutoModel

NUM_LABELS = 28

print(f'Loading model from {MODEL_NAME}...')
base_model = AutoModel.from_pretrained(MODEL_NAME)
hidden_size = base_model.config.hidden_size
print(f'Model loaded. Hidden size: {hidden_size}')

# Classification head đơn giản: [CLS] token -> linear -> 28 logits
classifier = nn.Linear(hidden_size, NUM_LABELS)

base_model = base_model.to(DEVICE)
classifier = classifier.to(DEVICE)

# Forward pass với 2 câu mẫu
sample_texts = test_sentences[:2]
enc = tokenizer(sample_texts, return_tensors='pt', truncation=True,
                max_length=128, padding=True)
enc = {k: v.to(DEVICE) for k, v in enc.items()}

with torch.no_grad():
    outputs = base_model(**enc)
    cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
    logits = classifier(cls_output)

print(f'\nInput batch size: {len(sample_texts)}')
print(f'CLS output shape: {cls_output.shape}')
print(f'Logits shape    : {logits.shape}')   # phải là [2, 28]
print(f'Expected         : [{len(sample_texts)}, {NUM_LABELS}]')
print(f'Shape correct   : {logits.shape == torch.Size([len(sample_texts), NUM_LABELS])}')

Loading model from uitnlp/visobert...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 21245.45it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded. Hidden size: 768

Input batch size: 2
CLS output shape: torch.Size([2, 768])
Logits shape    : torch.Size([2, 28])
Expected         : [2, 28]
Shape correct   : True


In [6]:
# Cell 6: Mini training batch (3 batch, loss + backward)
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class SimpleEmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts  = df['text'].tolist()
        self.labels = df['labels'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        label_vec = torch.zeros(NUM_LABELS)
        for lbl in self.labels[idx]:
            label_vec[lbl] = 1.0
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels'        : label_vec
        }

# Chỉ dùng 16 mẫu đầu để test nhanh
small_ds = SimpleEmotionDataset(train.head(16), tokenizer)
loader   = DataLoader(small_ds, batch_size=4, shuffle=False)

# pos_weight theo công thức paper: (N - n_i) / n_i
N = len(train)
label_counts_all = torch.zeros(NUM_LABELS)
for labels in train['labels']:
    for lbl in labels:
        label_counts_all[lbl] += 1
pos_weight = (N - label_counts_all) / label_counts_all
pos_weight = pos_weight.to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(
    list(base_model.parameters()) + list(classifier.parameters()),
    lr=2e-5
)

base_model.train()
classifier.train()

print('Running 3 mini batches...\n')
for step, batch in enumerate(loader):
    if step >= 3:
        break

    input_ids  = batch['input_ids'].to(DEVICE)
    attn_mask  = batch['attention_mask'].to(DEVICE)
    labels_vec = batch['labels'].to(DEVICE)

    optimizer.zero_grad()
    outputs    = base_model(input_ids=input_ids, attention_mask=attn_mask)
    cls_out    = outputs.last_hidden_state[:, 0, :]
    logits     = classifier(cls_out)
    loss       = criterion(logits, labels_vec)
    loss.backward()
    optimizer.step()

    mem_mb = torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    print(f'Step {step+1}: loss={loss.item():.4f}  |  GPU mem used={mem_mb:.0f} MB')

print('\nBackward pass OK.')

Running 3 mini batches...

Step 1: loss=1.3125  |  GPU mem used=1508 MB
Step 2: loss=1.3906  |  GPU mem used=1508 MB
Step 3: loss=1.2254  |  GPU mem used=1509 MB

Backward pass OK.


In [7]:
# Cell 7: Metric smoke test + export JSON
import json, os
from sklearn.metrics import f1_score
import numpy as np

# Dùng random logits để test metric pipeline
np.random.seed(42)
dummy_logits = np.random.randn(32, NUM_LABELS)
dummy_probs  = 1 / (1 + np.exp(-dummy_logits))   # sigmoid
dummy_preds  = (dummy_probs >= 0.5).astype(int)
dummy_labels = (np.random.rand(32, NUM_LABELS) > 0.85).astype(int)

macro_f1 = f1_score(dummy_labels, dummy_preds, average='macro', zero_division=0)
print(f'Metric pipeline OK. Dummy Macro F1 (random): {macro_f1:.4f}')
print('(Số này không có nghĩa — chỉ để xác nhận sklearn chạy được)')

# Export
os.makedirs('../artifacts', exist_ok=True)

result = {
    'dataset_loaded'         : True,
    'train_rows'             : int(len(train)),
    'val_rows'               : int(len(val)),
    'test_rows'              : int(len(test)),
    'num_labels'             : len(all_ids),
    'neutral_present'        : (27 in all_ids),
    'mean_labels_per_sample' : round(float(train['labels'].apply(len).mean()), 4),
    'cuda_available'         : torch.cuda.is_available(),
    'gpu_name'               : torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'model_forward_pass'     : True,
    'logit_shape'            : [2, NUM_LABELS],
    'backward_pass'          : True,
    'metric_pipeline'        : True,
    'approx_gpu_mem_used_mb' : round(torch.cuda.memory_allocated() / 1024**2, 1) if torch.cuda.is_available() else None
}

with open('../artifacts/feasibility_check.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print('\n=== Feasibility Check Result ===')
for k, v in result.items():
    print(f'{k:<30}: {v}')

Metric pipeline OK. Dummy Macro F1 (random): 0.2436
(Số này không có nghĩa — chỉ để xác nhận sklearn chạy được)

=== Feasibility Check Result ===
dataset_loaded                : True
train_rows                    : 16531
val_rows                      : 2066
test_rows                     : 2067
num_labels                    : 28
neutral_present               : True
mean_labels_per_sample        : 1.9082
cuda_available                : True
gpu_name                      : NVIDIA GeForce RTX 4060 Laptop GPU
model_forward_pass            : True
logit_shape                   : [2, 28]
backward_pass                 : True
metric_pipeline               : True
approx_gpu_mem_used_mb        : 1508.5


In [8]:
# Cell 8: VRAM check với batch_size thực tế
torch.cuda.empty_cache()
loader_full = DataLoader(small_ds, batch_size=32, shuffle=False)
batch = next(iter(loader_full))
input_ids = batch['input_ids'].to(DEVICE)
attn_mask = batch['attention_mask'].to(DEVICE)
labels_vec = batch['labels'].to(DEVICE)

optimizer.zero_grad()
outputs = base_model(input_ids=input_ids, attention_mask=attn_mask)
cls_out = outputs.last_hidden_state[:, 0, :]
logits = classifier(cls_out)
loss = criterion(logits, labels_vec)
loss.backward()

mem_mb = torch.cuda.memory_allocated() / 1024**2
print(f'batch_size=32: GPU mem used = {mem_mb:.0f} MB / 8192 MB')
print(f'Headroom: {8192 - mem_mb:.0f} MB')


batch_size=32: GPU mem used = 1513 MB / 8192 MB
Headroom: 6679 MB
